In [ ]:
!pip install transformers datasets accelerate pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive is connected") #since we are using google colab

Mounted at /content/drive
Drive is connected


In [1]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorForSeq2Seq,
)
from sklearn.model_selection import train_test_split
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU found: {gpu_props.name}, VRAM: {gpu_props.total_memory / 1024**3:.2f} GB")
else:
    print("No GPU found, using CPU.")

GPU found: Tesla T4, VRAM: 14.74 GB


In [2]:
# --------------------------------------------------
# Model
# --------------------------------------------------
model_name = "Turkish-NLP/t5-efficient-small-MLSUM-TR-fine-tuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/839k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/569M [00:00<?, ?B/s]

In [7]:
# --------------------------------------------------
# Data
# --------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/AiProject/Datasets/cleaned_train_5000.csv"

df = pd.read_csv(DATA_PATH)[["cleaned_article", "cleaned_summary"]].dropna()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df)} samples.")

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

Loaded 5000 samples.


In [8]:
# --------------------------------------------------
# Tokenization
# --------------------------------------------------
max_input_length = 256
max_target_length = 64

def preprocess(examples):
    model_inputs = tokenizer(
        examples["cleaned_article"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=examples["cleaned_summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )

    # Replace padding token id's with -100 to ignore in loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
# --------------------------------------------------
# Training setup
# --------------------------------------------------

# Google Drive'a kaydetmek için çıkış dizinini güncelleyin
output_path = "/content/drive/MyDrive/AiProject/Models/t5_finetune_results"

training_args = TrainingArguments(
    output_dir=output_path,
    eval_strategy="steps",
    eval_steps=400,
    save_steps=400,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    fp16=False,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=callbacks,
)

/tmp/ipython-input-2590853027.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
# --------------------------------------------------
# Training
# --------------------------------------------------
print("Training has been started")
trainer.train()
print("Training is finished")

Training has been started


Step,Training Loss,Validation Loss
400,2.870600,2.475203
800,2.812400,2.408703
1200,2.750000,2.360553
1600,2.793400,2.317086
2000,2.892400,2.288381
2400,2.517400,2.276193
2800,2.546500,2.263357
3200,2.518900,2.258749
3600,2.452600,2.248633
4000,2.614400,2.243787


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Training is finished


In [13]:
# --------------------------------------------------
# Save model
# --------------------------------------------------

final_model_path = "/content/drive/MyDrive/AiProject/Models/t5_finetuned"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f" Model saved to {final_model_path}")

# --------------------------------------------------
# Example inference
# --------------------------------------------------
text = df["cleaned_article"].iloc[0]
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
summary_ids = model.generate(**inputs, max_length=64, num_beams=4)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Example summary:\n", summary)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


 Model saved to /content/drive/MyDrive/AiProject/Models/t5_finetuned
Example summary:
 hububat bakliyat yağlı tohumlar mamulleri ihracatı yılın ilk ayında geçen yılın aynı dönemine oranla yüzde lık artışla milyar dolara ulaştı
